### Analysis of Student Data

Examining student data to draw conclusion by testing hypothesis.  

It would seem best to target exam score and answer the question, what variables affect test score the most? With an accurate model, we can predict our most likely test score based how those variables express.

---  

### **Data Wrangling**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sts

pd.set_option("mode.copy_on_write", True)

df_data = Path().cwd().parent.parent/"Data"/"student_habits_performance.csv"
student_df = pd.read_csv(df_data)
student_df["parental_education_level"].unique()

In [ ]:
print(student_df.describe(include='all'))

student_df.head(5)

Based on this small exploration alone, we can see we have multiple categorical variables alongside numerous numerical values, both float and int.  

We also see some great cursory stats about our numerical data, giving us an idea of:  
- Data scale and volume
  - We have 1,000 values per column
  - There is not a massive difference across all our values (min=0, max=100).
- How that data is distributed across each feature.  

Looking at `student_df["parental_education_level"]`, we can also see there's missing data. To cnofirm that's the only column, let's run a for loop.

In [ ]:
missing_data = student_df.isnull()

for column in missing_data:
    print(missing_data[column].value_counts())

Now, to get a closer look at the unique value counts of that feature.  

In [ ]:
parental_education_level_counts = student_df[
    "parental_education_level"
].value_counts(dropna=False).to_frame()
parental_education_level_counts.columns = ['value_counts']
parental_education_level_counts.index.name = "parental_education_level"

print(parental_education_level_counts.reset_index())

There are 91 missing values out of 1,000--that's 9.1% of our data set.  

While perfectly fine to stop at replacing the missing values with  
the frequency, it could be worth while to create a **binary  
indicator column** for missing parental education to explore  
relationship with student performance. 

In [ ]:
missing_ed_df = student_df[["parental_education_level"]]
missing_ed_df = missing_ed_df.rename(columns={"parental_education_level": "missing_parent_ed"})

for ed_index in list(range(len(missing_ed_df))):
    if pd.isnull(student_df.loc[ed_index, "parental_education_level"]):
        missing_ed_df.loc[ed_index, "missing_parent_ed"] = 1
    else:
        missing_ed_df.loc[ed_index, "missing_parent_ed"] = 0

print(missing_ed_df.value_counts())



This data is now preserved in a separate data frame I can  
concatenate with the original, or a copy of the original,  
when it's time to model.  

I'm not sure removing these missing vals is the smart choice, or best one, so  
I will replace the Nan values with "Missing." Doing so will allow us to use  
correlation in the following analysis.

In [ ]:
student_df["parental_education_level"] = student_df[
    "parental_education_level"
].replace({np.nan: "Missing"})

student_df[["parental_education_level"]].value_counts()

This should have been one of the first things I did, but we will also check  
the data types of every variable in the dataframe. 

Fortunately, everything still checks out with the correct data type,  
regardless of whether or not we observed the "parental_education_level"  
variable *with* the Nan values. However, procedurally, it should have been done  
first.

In [ ]:
student_df.dtypes

--- 

### **Exploratory Data Analysis**

Beginning our EDA with a cool, refreshing correlation map. From there, we'll  
take a look at our categorical variables to determine relevancy to our target  
variable.

In [ ]:
corr = student_df.corr(numeric_only=True)

sns.heatmap(corr, cmap='coolwarm')

The above takes care of a majority of our numerical column analysis, which is  
pretty incredible considering we have immediate visual representation of all  
the columns we might want to include in our modeling.

**Relevant numerical cols**:  
- Study_hours_per_day  
- Mental_health_rating  
  - **Extremely minimal impact but might explain minute variation**:  
    - Exercise_frequency  
    - Sleep_frequency  
    - Attendance_percentage  

But what about an operation to analyze the `object` type variables en masse?  

For that, we need a more custom solution, but it will be applicable across  
other instances of analyzation in the future.  

In [ ]:
test_df = student_df.drop(columns="student_id") # unnecessary object column

for column in test_df.columns:
    if test_df[column].dtype == 'object':
        group_test = test_df[[column, "exam_score"]] # need ind and target vars
        print(f"{column} ANOVA: ")
        anova_list = [] # initialize empty list per cat var instance
        for unique_val in group_test[column].unique(): # access unique vals iter
            group = group_test.loc[
                group_test[column] == unique_val, "exam_score"
            ] # grouping exam scores by unique val instance
            anova_list.append(list(group))
        if len(anova_list) == len(group_test[column].unique()):
            anova_results = sts.f_oneway(*anova_list) # unpack list into fxn
            print(anova_results)
        else:
            print("\nYour failure is weakness; try again.")

**ANOVA results**  

Gender:  
- F-statistic: 0.14
- P-val: 0.87  

Part time job:  
- F-satistic: 0.70  
- P-val: 0.40  

Diet quality:  
- F-statistic: 1.2  
- P-val: 0.28  

parental education level:  
- F-statistic: 0.65  
- P-val: 0.58  

Internet quality:
- F-statistic: 1.46  
- P-val: 0.23  

Extracuricular participation:   
- F-statistic: 0.0007  
- P-val: 0.97  

Overall, the variation within these groups is not really larger than the  
variation between these groups, and **none** of them reject the null hypothesis.

---  
### **Data Modeling**  

This data is remarkably linear--most likely due to it being user-generated  
test data (nothing wrong with that). I doubt we'll need polynomial regression,  
ridge regression, scalling, or transformation.  

The  variables we want to focus on are:  
- `study_hours_per_day`  
- `mental_health_rating`  
- `exam_score`  

I will make two models though, to test for more variation in the hopes of  
achieving a more accurate model. Those additional variables will be:  
- `exercise_frequency`  
- `sleep_hours`  
- `attendance_percentage`  

In preparation for this modeling, our imports.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split,cross_val_predict, cross_val_score

**Splitting our data for testing and predicting**

In [ ]:
numeric_cols = [
    "study_hours_per_day",
    "mental_health_rating"
]

x_data = test_df[numeric_cols]
y_data = test_df["exam_score"]

x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    test_size=0.3,#standard split
    random_state=0# repeatability
)

mlr = LinearRegression()
mlr.fit(x_train, y_train)
yhat = mlr.predict(x_test)

numeric_cols1 = [
    "study_hours_per_day",
    "mental_health_rating",
    "exercise_frequency",
    "sleep_hours",
    "attendance_percentage"
]

x_data1 = test_df[numeric_cols1]

x_train1, x_test1, y_train1, y_test1 = train_test_split(
    x_data1,
    y_data,
    test_size=0.3,
    random_state=0
)

mlr1 = LinearRegression()
mlr1.fit(x_train1, y_train1)
yhat1 = mlr1.predict(x_test1)

---  
### **Model Evaluation**

In [ ]:
model_fit = mlr.score(x_train, y_train)
model_generalization = mlr.score(x_test, y_test)
print(f"Our slimmer model - R^2 score: {model_fit}, model gen: {model_generalization}\n")

model_fit1 = mlr1.score(x_train1, y_train1)
model_generalization1 = mlr1.score(x_test1, y_test1)
print(f"Our larger model - R^2 score: {model_fit1}, model gen: {model_generalization1}\n")

We will visually confirm whether the assumptions behind our regression are  
being met.

In [ ]:
residuals = y_test - yhat
residuals1 = y_test1 - yhat1

# multi-plot setup
# 1 row, 2 figures
fig = plt.figure()
axes1 = fig.add_subplot(2, 1, 1)
axes2 = fig.add_subplot(2, 1, 2)

# add plot
axes1.scatter(y_test, residuals)
axes1.set_xlabel("Actual Values")
axes1.set_ylabel("Residual Values")
axes1.set_title("Residual Residuals v.s. Fitted - MLR 1")
axes1.axhline(y=0, color="red", linestyle="--")

axes2.scatter(y_test1, residuals1)
axes2.set_xlabel("Actual Values")
axes2.set_ylabel("Residual Values")
axes2.set_title("Residuals v.s. Fitted - MLR 2")
axes2.axhline(y=0, color="red", linestyle="--")

# prevent label overlapping
fig.set_tight_layout(True)

plt.show()